# OASIS-2 Minimal Colab Training

This notebook is a clean Colab-first path for OASIS-2 training.
It clones the repo, installs dependencies, runs the OASIS-2 training pipeline with live logs, and prints the saved summary.

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Cerebrasensecloud')
OASIS2_BUNDLE_ROOT = DRIVE_ROOT / 'OASIS-2'
RUNTIME_ROOT = DRIVE_ROOT / 'backend_runtime'
RUN_NAME = 'oasis2_colab_improved_v1'

for name, path in {
    'DRIVE_ROOT': DRIVE_ROOT,
    'OASIS2_BUNDLE_ROOT': OASIS2_BUNDLE_ROOT,
    'RUNTIME_ROOT': RUNTIME_ROOT,
}.items():
    print(name, path.exists(), path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT True /content/drive/MyDrive/Cerebrasensecloud
OASIS2_BUNDLE_ROOT True /content/drive/MyDrive/Cerebrasensecloud/OASIS-2
RUNTIME_ROOT True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime


In [2]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path('/content/cerebrasense')
BACKEND_ROOT = REPO_ROOT / 'alz_backend'
REPO_URL = 'https://github.com/Billrichard209/cerebrasense.git'
REQUIRED_COMMIT = 'e577752'

def run_checked(cmd, *, cwd=None, label=None):
    print(f"RUNNING {label or cmd[0]}: {' '.join(cmd)}", flush=True)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, flush=True)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed ({label or cmd[0]}): {' '.join(cmd)}")
    return completed

for stale_root in [Path('/content/cerebrasense'), Path('/content/Cerebrasense-')]:
    if stale_root.exists():
        shutil.rmtree(stale_root)

run_checked(['git', 'clone', REPO_URL, str(REPO_ROOT)], cwd='/content', label='git-clone')
run_checked(['git', 'checkout', 'main'], cwd=REPO_ROOT, label='git-checkout-main')
run_checked(['python3', '-m', 'pip', 'install', '-r', str(BACKEND_ROOT / 'requirements-colab.txt')], cwd=REPO_ROOT, label='pip-install')

repo_commit = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, label='git-rev-parse').stdout.strip()
print('repo_commit =', repo_commit)
print('required_commit =', REQUIRED_COMMIT)




RUNNING git-clone: git clone https://github.com/Billrichard209/cerebrasense.git /content/cerebrasense
Cloning into '/content/cerebrasense'...

RUNNING git-checkout-main: git checkout main
Your branch is up to date with 'origin/main'.

Already on 'main'

RUNNING pip-install: python3 -m pip install -r /content/cerebrasense/alz_backend/requirements-colab.txt

RUNNING git-rev-parse: git rev-parse HEAD
5dfbed5f49387b228c3fd576a7f93565f6b38ee2

repo_commit = 5dfbed5f49387b228c3fd576a7f93565f6b38ee2
required_commit = e577752


In [7]:
from pathlib import Path
import pandas as pd

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
print('run_root =', run_root.exists(), run_root)

for p in [
    run_root / 'metrics' / 'epoch_metrics.csv',
    run_root / 'checkpoints' / 'best_model.pt',
    run_root / 'checkpoints' / 'last_model.pt',
    run_root / 'reports' / 'colab_run_summary.json',
]:
    print(p, '->', p.exists())

metrics = run_root / 'metrics' / 'epoch_metrics.csv'
if metrics.exists():
    df = pd.read_csv(metrics)
    print(df.tail())



run_root = True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2/metrics/epoch_metrics.csv -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2/checkpoints/best_model.pt -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2/checkpoints/last_model.pt -> True
/content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2/reports/colab_run_summary.json -> False
   epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
0      1         0.0001    0.711397  0.731574  0.407407  0.417010   0.333333   
1      2         0.0001    0.693674  0.727824  0.481481  0.419753   0.461538   

     recall        f1  sensitivity  specificity  train_batches  val_batches  
0  0.185185  

In [ ]:
import subprocess
import sys
from pathlib import Path
import torch

print('cuda_available =', torch.cuda.is_available())
print('gpu_name =', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached to this Colab runtime. Stop here and switch Runtime -> Change runtime type -> GPU.')

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
last_checkpoint = run_root / 'checkpoints' / 'last_model.pt'
resume_args = []
if last_checkpoint.exists():
    print('Resuming from existing checkpoint:', last_checkpoint, flush=True)
    resume_args = ['--resume-from', str(last_checkpoint)]

cmd = [
    sys.executable,
    'scripts/train_oasis2_colab.py',
    '--project-root', str(BACKEND_ROOT),
    '--runtime-root', str(RUNTIME_ROOT),
    '--bundle-root', str(OASIS2_BUNDLE_ROOT),
    '--run-name', RUN_NAME,
    '--epochs', '30',
    '--batch-size', '2',
    '--gradient-accumulation-steps', '4',
    '--num-workers', '2',
    '--image-size', '96', '96', '96',
    '--learning-rate', '3e-4',
    '--weight-decay', '0.01',
    '--scheduler', 'cosine',
    '--step-size', '30',
    '--weighted-sampling',
    '--seed', '42',
    '--split-seed', '42',
    '--device', 'auto',
] + resume_args

print('RUNNING:', ' '.join(cmd), flush=True)
process = subprocess.Popen(
    cmd,
    cwd=BACKEND_ROOT,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'OASIS-2 training runner failed with exit code {return_code}')

print('OASIS-2 training runner completed successfully.', flush=True)



In [5]:
from pathlib import Path
import json
import pandas as pd

run_root = RUNTIME_ROOT / 'outputs' / 'runs' / 'oasis2' / RUN_NAME
summary_path = run_root / 'reports' / 'colab_run_summary.json'
metrics_path = run_root / 'metrics' / 'epoch_metrics.csv'

print('run_root =', run_root.exists(), run_root)
print('summary_path =', summary_path, '->', summary_path.exists())
print('metrics_path =', metrics_path, '->', metrics_path.exists())

if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2))
    print('training_ready =', summary.get('training_ready'))
    print('training_started =', summary.get('training_started'))
    print('blocked_reason =', summary.get('blocked_reason'))
    print('run_root =', summary.get('run_root'))
    print('best_checkpoint =', summary.get('best_checkpoint'))
else:
    print('No completed colab_run_summary.json yet. Showing current epoch metrics instead.')

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(metrics.tail())



run_root: True /content/drive/MyDrive/Cerebrasensecloud/backend_runtime/outputs/runs/oasis2/oasis2_colab_baseline_v2
epoch_metrics.csv True
best_model.pt True
last_model.pt True
   epoch  learning_rate  train_loss  val_loss  accuracy     auroc  precision  \
0      1         0.0001    0.711397  0.731574  0.407407  0.417010   0.333333   
1      2         0.0001    0.693674  0.727824  0.481481  0.419753   0.461538   

     recall        f1  sensitivity  specificity  train_batches  val_batches  
0  0.185185  0.238095     0.185185     0.629630            259           54  
1  0.222222  0.300000     0.222222     0.740741            259           54  


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')



True
Tesla T4
